# Manual vs automatic smile detection


In [ ]:
from notebook_setup import init
FINAL_DIR, PDF_DIR = init()
import warnings

import pandas as pd
from sklearn.metrics import roc_auc_score

from analysis_config import BASE_PATH, CONFIDENCE_THRESHOLD, MANUAL_DIR, SMILE_AUS
from manual_smile_utils import load_manual_entries, load_synced_session

warnings.filterwarnings("ignore")

for path, label in [(MANUAL_DIR, "manual coding"), (BASE_PATH, "OpenFace")]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label} dir: {path}")

print(f"OpenFace: {BASE_PATH}")
print(f"Manual coding: {MANUAL_DIR}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")

In [ ]:
manual_entries = load_manual_entries()
print(f"Loaded manual coding files: {len(manual_entries)}")


In [ ]:
session_stats = []
synced_frames = []

for entry in manual_entries:
    df_of = load_synced_session(entry)
    if df_of is None:
        continue

    session_id = entry["diti_id"]
    meeting_id = "_".join(session_id.split("_")[:2])
    y_true = df_of["manual_smile"].values
    composite = df_of["composite"].values
    au06 = df_of[SMILE_AUS[0]].values
    au12 = df_of[SMILE_AUS[1]].values

    session_stats.append({
        "session_id": session_id,
        "meeting_id": meeting_id,
        "n_frames": len(df_of),
        "auc_composite": roc_auc_score(y_true, composite),
        "auc_au06": roc_auc_score(y_true, au06),
        "auc_au12": roc_auc_score(y_true, au12),
    })
    synced_frames.append(df_of.assign(session_id=session_id, meeting_id=meeting_id))

df_stats = pd.DataFrame(session_stats)
df_all = pd.concat(synced_frames, ignore_index=True) if synced_frames else pd.DataFrame()
print(f"Valid sessions: {len(df_stats)}, meetings: {df_stats['meeting_id'].nunique() if len(df_stats) else 0}")

if len(df_stats) == 0:
    raise RuntimeError("No sessions passed filtering.")

pooled_auc = roc_auc_score(df_all["manual_smile"], df_all["composite"])


## Results

In [ ]:
results = pd.DataFrame({
    "metric": [
        "ROC-AUC (AU06+AU12)/2, pooled",
        "ROC-AUC (AU06+AU12)/2, mean per session",
        "ROC-AUC AU06, mean per session",
        "ROC-AUC AU12, mean per session",
        "Sessions",
        "Meetings",
        "Frames",
        "Manual smile frames (%)",
    ],
    "value": [
        f"{pooled_auc:.4f}",
        f"{df_stats['auc_composite'].mean():.4f}",
        f"{df_stats['auc_au06'].mean():.4f}",
        f"{df_stats['auc_au12'].mean():.4f}",
        len(df_stats),
        df_stats["meeting_id"].nunique(),
        len(df_all),
        f"{int(df_all['manual_smile'].sum())} ({100 * df_all['manual_smile'].mean():.1f}%)",
    ],
})
display(results)
